# 01 - Classical Transformer Benchmark (Track A)

**Purpose.** Train and evaluate DeiT-Small, Swin-Tiny and MaxViT-Tiny (plus optional extras) with one data pipeline, one split manifest, one loss and one evaluation protocol. Model selection uses the validation split only.

**Requires (inputs).** Output of notebook 00 (split manifest) for the same MODE; RSNA data; optional offline timm weights dataset.

**Produces (outputs, under `/kaggle/working/rsna_ich_artifacts/<MODE>/`).** `experiments/A_<model>/` (best/last checkpoints, history, val/test predictions, thresholds, metrics, CIs), `selection/best_transformer.json`, optional ablation H `experiments/H_<model>_brain3/`.

> Research software. Outputs are model predictions for a research study and **not** clinical diagnoses.
> Results in this notebook are only valid for the MODE they were produced in (`smoke` results check correctness and say nothing about performance).

In [ ]:
# ==== Setup: locate the project code and make `src` importable ====================================
# Works on Kaggle (project code attached as a Dataset, possibly nested after zip extraction) and locally.
import os, sys, importlib
from pathlib import Path

def find_project_root(max_depth: int = 5) -> Path:
    def ok(p: Path) -> bool:
        return (p / "src" / "__init__.py").is_file() and (p / "configs" / "data_config.yaml").is_file()
    cands = []
    if os.environ.get("RSNA_ICH_PROJECT_ROOT"):
        cands.append(Path(os.environ["RSNA_ICH_PROJECT_ROOT"]))
    cands += [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/RSNA_ICH_Research")]
    for c in cands:
        if ok(c):
            return c.resolve()
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not base.exists():
            continue
        stack = [(base, 0)]
        while stack:
            d, depth = stack.pop()
            if ok(d):
                return d.resolve()
            if depth < max_depth:
                try:
                    stack += [(p, depth + 1) for p in d.iterdir() if p.is_dir() and not p.name.startswith("stage_2_")]
                except OSError:
                    pass
    raise FileNotFoundError(
        "Project code not found. On Kaggle: 'Add Input' -> the dataset containing RSNA_ICH_Research "
        "(with src/ and configs/). Locally: run the notebook from RSNA_ICH_Research/notebooks or set "
        "the RSNA_ICH_PROJECT_ROOT environment variable.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
for _m in ["src", "src.utils.context", "src.data", "src.models", "src.quantum", "src.training", "src.evaluation"]:
    importlib.import_module(_m)  # fails loudly here if a dependency is missing
import src
print(f"Project root: {PROJECT_ROOT}  |  src version {src.__version__}")

In [ ]:
# ==== Optional dependencies ======================================================================
# Kaggle images ship torch, timm, pydicom, scikit-learn and OpenCV. PennyLane is only needed for the
# reference quantum backend (the default torch simulator does not need it). With internet enabled:
#   !pip install -q pennylane
# Without internet, attach a dataset of wheels and run: !pip install --no-index --find-links /kaggle/input/<wheels> pennylane
from src.utils.runtime import package_versions
print(package_versions())

In [ ]:
# ==== Configuration ================================================================================
# MODE: "smoke" (correctness check, minutes) | "pilot" (representative subset) | "full" (largest feasible).
MODE = "smoke"
# Optional per-config overrides without editing YAML, e.g. {"transformer": {"train": {"epochs": 2}}}
OVERRIDES = {}
import json
if os.environ.get("RSNA_ICH_OVERRIDES_JSON"):          # test hook used by tools/run_notebooks_synthetic.py
    OVERRIDES = json.loads(os.environ["RSNA_ICH_OVERRIDES_JSON"])
MODE = os.environ.get("RSNA_ICH_MODE", MODE)

from src.utils.context import ProjectContext
ctx = ProjectContext(PROJECT_ROOT, MODE, OVERRIDES)
print(ctx.describe())
print("Installed packages:", ctx.runtime.packages)
ctx.save_run_info("01_transformers")

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display
from src import LABELS
from src.utils.io import atomic_write_json, atomic_save_dataframe, read_json, dataframe_to_markdown
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

## 1. Load the shared split manifest and configuration

In [ ]:
manifest, man_info = ctx.manifest()
ctx.restore_experiments(("A_", "H_"))  # resume interrupted runs from an attached previous version
tcfg = ctx.cfg["transformer"]
print("manifest", man_info["manifest_hash"][:12], "| grouping:", man_info["grouping_level"], "| images:", len(manifest))
print("Runtime:", ctx.runtime.summary())
models = list(tcfg["models"]) + [m for m in tcfg.get("optional_models", []) if m["name"] in tcfg.get("run_optional", [])]
print("Planned models:", [m["name"] for m in models])

## 2. Train / evaluate each model (restartable)
A failure (missing weights, incompatible timm, OOM) is recorded and the loop continues. Completed experiments with an identical configuration are loaded, not retrained. Weight status is reported for every model.

In [ ]:
from src.training.experiment import run_image_experiment
results = {}
for m in models:
    m = {**m, "weights_dir": tcfg.get("weights_dir")}
    exp_id = f"A_{m['name']}"
    print("=" * 100, "\n", exp_id)
    results[exp_id] = run_image_experiment(
        ctx, exp_id, m, tcfg["train"], tcfg["loss"], manifest=manifest, manifest_info=man_info,
        preprocess=ctx.cfg["data"]["preprocessing"], augment=ctx.cfg["data"]["augmentation"], limits=tcfg["limits"],
        loader_cfg=tcfg["loader"], threshold_method=tcfg["threshold_method"], n_boot=tcfg["n_boot"], seed=tcfg["seed"])

## 3. Benchmark table (sorted by **validation** macro ROC-AUC; test columns are reported, never used for selection)

In [ ]:
def row(exp_id, r):
    out = {"experiment": exp_id, "status": r.get("status"), "weights": (r.get("build") or {}).get("status"),
           "params_M": round((r.get("build") or {}).get("n_params", 0) / 1e6, 1)}
    if r.get("status") == "succeeded":
        ci = r.get("test_macro_auc_ci", {})
        out.update(val_macro_auc=r["val_metrics"]["macro_roc_auc"], test_macro_auc=r["test_metrics"]["macro_roc_auc"],
                   test_ci=f"[{ci.get('ci_low', np.nan):.3f}, {ci.get('ci_high', np.nan):.3f}]",
                   test_macro_ap=r["test_metrics"]["macro_ap"], test_any_auc=r["test_metrics"].get("any_roc_auc"),
                   best_epoch=r["fit"]["best_epoch"], epochs_run=r["fit"]["epochs_run"], train_min=round(r["seconds_total"] / 60, 1))
    else:
        out["reason"] = r.get("error") or r.get("reason")
    return out
bench = pd.DataFrame([row(k, v) for k, v in results.items()])
if "val_macro_auc" in bench:
    bench = bench.sort_values("val_macro_auc", ascending=False, na_position="last")
display(bench)
atomic_save_dataframe(bench, ctx.dir("analysis") / "trackA_benchmark.csv")

## 4. Per-label results, training curves and ROC/PR curves (held-out test, frozen validation thresholds)

In [ ]:
from src.evaluation.visualization import plot_history, plot_roc_pr
from src.evaluation.predictions import load_predictions, arrays
for exp_id, r in results.items():
    if r.get("status") != "succeeded":
        continue
    d = ctx.dir("experiments", exp_id)
    display(pd.read_csv(d / "metrics_test_per_label.csv")[["label", "n_pos", "roc_auc", "average_precision", "sensitivity",
                                                           "specificity", "precision", "f1", "brier", "threshold", "undefined"]].round(4))
    plot_history(pd.read_csv(d / "history.csv"), d / "history.png", exp_id); plt.show()
    pdf, _ = load_predictions(d / "predictions_test.parquet", man_info["manifest_hash"])
    p, t = arrays(pdf)
    plot_roc_pr(t, p, d / "roc_pr_test.png", f"{exp_id} - held-out test"); plt.show()

## 5. Select the best Transformer by validation macro ROC-AUC

In [ ]:
from src.training.experiment import select_best_by_validation
best = select_best_by_validation(ctx, prefixes=("A_",))
if best is None:
    print("No Track-A experiment succeeded - nothing to select.")
else:
    sel = {"experiment_id": best["experiment_id"], "selected_by": "validation macro ROC-AUC",
           "val_macro_auc": best["val_metrics"]["macro_roc_auc"], "manifest_hash": man_info["manifest_hash"],
           "weights_status": best["build"]["status"]}
    atomic_write_json(ctx.dir("selection") / "best_transformer.json", sel)
    print(sel)

## 6. Ablation H - single-window vs multi-window input (best Transformer, identical budget)

In [ ]:
if tcfg.get("ablation_single_window") and best is not None:
    mcfg = dict(best["config"]["model"]); mcfg.pop("in_chans", None)
    pp_single = {**ctx.cfg["data"]["preprocessing"], "channel_mode": "brain3"}
    exp_id = f"H_{mcfg['name']}_brain3"
    rH = run_image_experiment(ctx, exp_id, mcfg, tcfg["train"], tcfg["loss"], manifest=manifest, manifest_info=man_info,
                              preprocess=pp_single, augment=ctx.cfg["data"]["augmentation"], limits=tcfg["limits"],
                              loader_cfg=tcfg["loader"], threshold_method=tcfg["threshold_method"], n_boot=tcfg["n_boot"],
                              seed=tcfg["seed"])
    display(pd.DataFrame([row(best["experiment_id"] + " (bsb)", best), row(exp_id, rH)]))
else:
    ctx.registry.record("H_single_window", "skipped", reason="disabled in this MODE or no Track-A model succeeded")
    print("Ablation H skipped in this mode (recorded in the registry).")

**Interpretation guidance.** Compare models on validation for selection; the test column with patient-level CIs is for reporting. Overlapping CIs mean the data cannot distinguish the models. Paired tests are in notebook 04.